In [12]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import numpy as np
import matplotlib.pyplot as plt
import meshio
from pathlib import Path
from Train_fun import train_fun
from scipy.interpolate import Rbf
from RK import solve_rk4_adaptive_fixed_dt
from RK import build_f_sparse_mixed
from Bulid_Library import build_polynomial_library
from SSSR import SSSR
from SSSR import linear_reg

In [13]:
def rrmse(x,y):
    return (np.mean((x-y)**2))**0.5/(np.mean(y**2))**0.5

In [14]:
def read_snapshots_one_parameter(
    param_id,
    input_dir="NS_data",
    param_folder_pattern="param_{pid:03d}",
    file_pattern="solution_*.vtu",
    field_name="u",
):

    input_dir = Path(input_dir)
    param_dir = input_dir / param_folder_pattern.format(pid=param_id)
    vtu_files = sorted(param_dir.glob(file_pattern))
    snapshots = []
    for file in vtu_files:
        mesh = meshio.read(file)
        u = np.asarray(mesh.point_data[field_name]).reshape(-1)
        snapshots.append(u)
    U = np.vstack(snapshots)

    return U

# Data generation

In [187]:
U_all = []
Para_number = 11
Para = np.array(range(Para_number))*(10**-5*5) + 0.0005
for i in range(Para_number):
    U_ori =  read_snapshots_one_parameter(param_id=i*5)
    U_all.append(U_ori)

U_total = np.vstack(U_all)

In [462]:
U_total = U_total
Library = []
start_id = 150
for i in range(Para_number):
    cache = U_total[i*351+start_id:(i+1)*351]
    Library.append(cache)
U = np.vstack(Library)
m = cache.shape[0]

# POD

In [449]:
U_mean = np.mean(U, axis=0)
U_centered = U - U_mean
U_snapshots = U_centered.T  # 现在 shape 是 (n, m)，每列是一个 snapshot
Phi, Sigma, Vt = np.linalg.svd(U_snapshots, full_matrices=False)

In [450]:
latent_dim = 8
Phi_r = Phi[:,:latent_dim]
Z = np.transpose(np.dot(Phi_r.T, U_snapshots))  # shape = (m, r)
U_reconstructed = np.dot(Z, Phi_r.T) + U_mean
print('reconstruct error:', rrmse(U_reconstructed,U))

reconstruct error: 0.006185605313746178


# SSSR

In [451]:
r = Z.shape[1]
dt = 0.02

dZ = np.zeros_like(Z)

for i in range(Para_number):
    start = i * m
    end   = (i + 1) * m
    a = Z[start:end, :]  
    da = np.empty_like(a)

    da[0] = (-25*a[0] + 48*a[1] - 36*a[2] + 16*a[3] - 3*a[4]) / (12*dt)
    da[1] = (-3*a[0] - 10*a[1] + 18*a[2] - 6*a[3] + a[4]) / (12*dt)
    da[2:-2] = (-a[4:] + 8*a[3:-1] - 8*a[1:-3] + a[0:-4]) / (12*dt)
    da[-2] = (-a[-5] + 6*a[-4] - 18*a[-3] + 10*a[-2] + 3*a[-1]) / (12*dt)
    da[-1] = (3*a[-5] - 16*a[-4] + 36*a[-3] - 48*a[-2] + 25*a[-1]) / (12*dt)

    dZ[start:end, :] = da

In [452]:
include_functions = False
include_bias = False
degree = 1
Theta, feature_names = build_polynomial_library(Z, degree=degree, include_bias=include_bias, include_functions=include_functions)
sparsity_level = 8
Z_pred = Z.copy()
coef_matrix = np.zeros((Para_number,(sparsity_level+1)*latent_dim))
Supports = np.zeros((latent_dim,sparsity_level))

for k in range(latent_dim):
    loss = 0
    support = SSSR(Theta, dZ[:,k].reshape(-1,1), sparsity_level=sparsity_level, Para_number=Para_number)
    print('Support set for latent variable {}:'.format(k), support)
    Supports[k,:] = support
    Library = []
    for j in range(Para_number):
        Target = dZ[m*j:m*(j+1),k].reshape(-1,1)
        state = Z[m*j:m*(j+1),k].reshape(-1,1)
        Feature = Theta[m*j:m*(j+1),support]
        reg, pred = linear_reg(Feature,Target)
        coef_matrix[j,(sparsity_level+1)*k] = reg.intercept_
        coef_matrix[j,(sparsity_level+1)*k+1:(sparsity_level+1)*(k+1)] = reg.coef_
        loss = loss + rrmse(pred,Target)
        state_next_pred = state + dt * pred
        state_pred = state.copy()
        state_pred[1:] = state_next_pred[:-1]
        Library.append(state_pred)
    print('Regression error of latent variable {}:'.format(k), loss/Para_number)
    state_pred = np.vstack(Library)
    Z_pred[:,k] = np.squeeze(state_pred)
    Supports = Supports.astype('int')

Support set for latent variable 0: [1, 6, 4, 7, 3, 5, 0, 2]
Regression error of latent variable 0: 0.002968980685370203
Support set for latent variable 1: [0, 1, 4, 7, 6, 3, 5, 2]
Regression error of latent variable 1: 0.0021665086505024047
Support set for latent variable 2: [0, 1, 4, 3, 2, 5, 7, 6]
Regression error of latent variable 2: 0.05390506817142069
Support set for latent variable 3: [4, 2, 7, 5, 3, 0, 1, 6]
Regression error of latent variable 3: 0.006717440558120286
Support set for latent variable 4: [3, 5, 1, 4, 0, 7, 2, 6]
Regression error of latent variable 4: 0.006460080522302568
Support set for latent variable 5: [0, 3, 4, 7, 6, 5, 1, 2]
Regression error of latent variable 5: 0.07212845299495828
Support set for latent variable 6: [1, 6, 7, 5, 0, 3, 4, 2]
Regression error of latent variable 6: 0.061207497905689995
Support set for latent variable 7: [6, 5, 3, 4, 7, 1, 0, 2]
Regression error of latent variable 7: 0.06743722133488245


In [453]:
Library = []
for j in range(Para_number):
    f, info = build_f_sparse_mixed(feature_names , Supports=Supports, 
                              coef_matrix=coef_matrix[j].reshape(latent_dim,-1), latent_dim=latent_dim)
    z0 =  Z[m*j,:]
    T, Y = solve_rk4_adaptive_fixed_dt(f, 0, dt*(m-1), z0, dt, rtol=1e-6, atol=1e-9, h0=1e-2, h_min=0.00001)
    Library.append(Y)
Z_pred_multi_step = np.vstack(Library)

In [454]:
print('The prediction error of Z by multiple step:', rrmse(Z_pred_multi_step,Z))
U_pred_multi_step = np.dot(Z_pred_multi_step, Phi_r.T) + U_mean  # shape = (m, n)
print('The prediction error of X by multiple step:', rrmse(U_pred_multi_step,U)) 

The prediction error of Z by multiple step: 0.06423114429424269
The prediction error of X by multiple step: 0.008440639384532773


In [455]:
error0 = np.zeros((Para_number)) 
for j in range(Para_number):
    loss = rrmse(U_pred_multi_step[k*m:(k+1)*m,],U[k*m:(k+1)*m])
    print(j,'Mu:',Para[j],'The prediction error of X by multiple step:', loss) 

0 Mu: 0.0005 The prediction error of X by multiple step: 0.008043031030705597
1 Mu: 0.00055 The prediction error of X by multiple step: 0.008043031030705597
2 Mu: 0.0006000000000000001 The prediction error of X by multiple step: 0.008043031030705597
3 Mu: 0.00065 The prediction error of X by multiple step: 0.008043031030705597
4 Mu: 0.0007 The prediction error of X by multiple step: 0.008043031030705597
5 Mu: 0.00075 The prediction error of X by multiple step: 0.008043031030705597
6 Mu: 0.0008 The prediction error of X by multiple step: 0.008043031030705597
7 Mu: 0.0008500000000000001 The prediction error of X by multiple step: 0.008043031030705597
8 Mu: 0.0009 The prediction error of X by multiple step: 0.008043031030705597
9 Mu: 0.0009500000000000001 The prediction error of X by multiple step: 0.008043031030705597
10 Mu: 0.001 The prediction error of X by multiple step: 0.008043031030705597


# Parameter-to-coefficient mapping

In [456]:
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.interpolate import Rbf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [457]:
Para_data = np.hstack((Para.reshape(-1,1),coef_matrix))

## RBF

In [458]:
X = Para_data[:, 0:1].reshape(-1,1)        
Y = Para_data[:, 1:] 
N, r = Y.shape

rbf_models = []
for j in range(r):
    rbf = Rbf(X[:,0], Y[:, j], function='multiquadric')
    rbf_models.append(rbf)

## NN

In [472]:
class FourNet_sin(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, 128)
        self.linear2 = nn.Linear(128, 64)
        self.linear3 = nn.Linear(64,64)
        self.linear4 = nn.Linear(64,output_dim)

    def forward(self, x):
        x = torch.sin(self.linear1(x))
        x = torch.sin(self.linear2(x))
        x = torch.sin(self.linear3(x))
        x = self.linear4(x)
        return x

In [473]:
X = Para_data[:, 0:1]        
Y = Para_data[:, 1:]        

Features= [np.arange(19),np.arange(19,36),np.arange(36,Y.shape[1])]
X_train_t = torch.from_numpy(X).float().to(device)
NN_model,Loss = [], []
for feature in Features:
    Y_train_t = torch.from_numpy(Y[:,feature]).float().to(device)
    output_dim = Y_train_t.shape[1]
    model = FourNet_sin(input_dim=1, output_dim=output_dim).to(device)
    min_loss,runing_time,loss_history = train_fun(model,X_train_t,Y_train_t,N_red_lr=4,epochs=15000,lr=0.001,threshold=-1,printfun=False)
    NN_model.append(model)
    Loss.append(loss_history)

运行时间： 77.99355959892273
loss: 0.032878804951906204
运行时间： 75.0403344631195
loss: 0.011593202129006386
运行时间： 75.84248232841492
loss: 0.0213424414396286


# Online predict

## Set test parameter point

In [476]:
Para_test_id = 32
Para_test = np.array([[Para_test_id*10**-5+0.0005]])
Para_test_number = len(Para_test)
U_test =  read_snapshots_one_parameter(param_id=Para_test_id)[start_id:]

In [477]:
Para_test

array([[0.00082]])

In [478]:
# get latent initial condiction
U_centered_test = U_test - U_mean
U_snapshots_test = U_centered_test.T 
Z_test = np.dot(Phi_r.T, U_snapshots_test).T

## SSSR-RBF

In [479]:
# predcit the coefficients
Y_test_pred_RBF = []
for j, rbf in enumerate(rbf_models):
    Y_test_pred_RBF.append(rbf(Para_test.reshape(-1,1)))
Y_test_pred_RBF = np.hstack(Y_test_pred_RBF)

# sloving the latent dynamical system 
Library = []
for j in range(Para_test_number):   
    f, info = build_f_sparse_mixed(feature_names , Supports=Supports, 
                              coef_matrix=Y_test_pred_RBF[j].reshape(latent_dim,-1), latent_dim=latent_dim)
    z0 =  Z_test[m*j,:]
    T, Y = solve_rk4_adaptive_fixed_dt(f, 0, dt*(m-1), z0, dt, rtol=1e-6, atol=1e-9, h0=1e-2, h_min=0.001)
    Library.append(Y)
Z_pred_multi_step_RBF = np.vstack(Library)

# reconstruction 
U_pred_multi_step_RBF = np.dot(Z_pred_multi_step_RBF, Phi_r.T) + U_mean  # shape = (m, n)

In [480]:
rrmse(U_pred_multi_step_RBF,U_test)

0.007072529671293613

## SSSR_NN

In [481]:
# predcit the coefficients
X_test = Para_test.reshape(-1,1)
X_test_t = torch.from_numpy(X_test).float().to(device)
Y_test_pred_NN = []
model.eval()
with torch.no_grad():
    for model in NN_model:
        Y_test_pred_NN.append(model(X_test_t).cpu().numpy())
Y_test_pred_NN = np.hstack(Y_test_pred_NN)

# sloving the latent dynamical system 
Library = []
for j in range(Para_test_number):   
    f, info = build_f_sparse_mixed(feature_names , Supports=Supports, 
                              coef_matrix=Y_test_pred_NN[j].reshape(latent_dim,-1), latent_dim=latent_dim)
    z0 =  Z_test[m*j,:]
    T, Y = solve_rk4_adaptive_fixed_dt(f, 0, dt*(m-1), z0, dt, rtol=1e-6, atol=1e-9, h0=1e-2, h_min=0.001)
    Library.append(Y)
Z_pred_multi_step_NN = np.vstack(Library)

# reconstruction 
U_pred_multi_step_NN= np.dot(Z_pred_multi_step_NN, Phi_r.T) + U_mean  # shape = (m, n)

In [482]:
rrmse(U_pred_multi_step_NN,U_test)

0.007824696293771652